### Distribution of $S-C$ for various voting rules using Scottish data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import fsolve
from functools import partial
from matplotlib.lines import Line2D 
from pathlib import Path
import math
import os
from glob import glob
from votekit.cvr_loaders import load_scottish
from votekit.utils import first_place_votes
from votekit.elections import STV, Borda, SNTV, RankedPairs, BlocPlurality, Plurality

SCOT_ELEX_PATH = "../../../scot-elex"
# SCOT_ELEX_PATH = "/Users/cdonnay/PycharmProjects/scot-elex"

b_bloc_parties = ["Scottish National Party (SNP)", "Green (Gr)"]
b_bloc_label = "SNP and Green"
bloc_order = "AB"
stv_color = "#1560BD"
borda_color = "#FB607F"

In [ ]:
all_files = glob(f"{SCOT_ELEX_PATH}/*_cands/*.csv")
no_b_bloc_parties = 0

data_stv = {1:[], 2:[], 3:[], 4:[], 5:[]}
data_borda = {1:[], 2:[], 3:[], 4:[], 5:[]}
data_ranked_pairs = {1:[], 2:[], 3:[], 4:[], 5:[]}
data_sntv = {1:[], 2:[], 3:[], 4:[], 5:[]}
block_plurality_data = {1:[], 2:[], 3:[], 4:[], 5:[]}


for file_name in all_files:
    profile, num_seats, cand_list, cand_to_party, ward_name = load_scottish(file_name)
    cand_to_bloc = {c:"B" if cand_to_party[c] in b_bloc_parties 
                else "A" for c in cand_list}

    bloc_to_cand_num = {"A": len([c for c, bloc in cand_to_bloc.items() if bloc == "A"]),
                    "B": len([c for c, bloc in cand_to_bloc.items() if bloc == "B"])}

    

    fpv_dict = first_place_votes(profile)

    total_fpv = sum(fpv_dict.values())
    fpv_b_bloc = sum([v for c,v in fpv_dict.items() if cand_to_bloc[c] == "B"])
    fpv_b_share = fpv_b_bloc/total_fpv

    if bloc_to_cand_num["B"] < math.floor(fpv_b_share*num_seats) or bloc_to_cand_num["B"] == 0:
        continue 


    stv_election  = STV(profile, m = num_seats)
    stv_winners= [c for s in stv_election.get_elected() for c in s]
    stv_num_B_winners = len([c for c in stv_winners if cand_to_bloc[c] == "B"])
    stv_b_seat_share = stv_num_B_winners/num_seats
    data_stv[num_seats].append((fpv_b_share, stv_b_seat_share))

    borda_election  = Borda(profile, m = num_seats, tiebreak="first_place")
    borda_winners= [c for s in borda_election.get_elected() for c in s]
    borda_num_B_winners = len([c for c in borda_winners if cand_to_bloc[c] == "B"])
    borda_b_seat_share = borda_num_B_winners/num_seats
    data_borda[num_seats].append((fpv_b_share, borda_b_seat_share))

    ranked_pairs_election = RankedPairs(profile, m = num_seats)
    ranked_pairs_winners= [c for s in ranked_pairs_election.get_elected() for c in s]
    ranked_pairs_num_B_winners = len([c for c in ranked_pairs_winners if cand_to_bloc[c] == "B"])
    ranked_pairs_b_seat_share = ranked_pairs_num_B_winners/num_seats
    data_ranked_pairs[num_seats].append((fpv_b_share, ranked_pairs_b_seat_share))

    sntv_election = SNTV(profile, m = num_seats, tiebreak='borda')
    sntv_winners = [c for s in sntv_election.get_elected() for c in s]
    sntv_num_B_winners = len([c for c in sntv_winners if cand_to_bloc[c] == "B"])
    sntv_b_seat_share = sntv_num_B_winners/num_seats
    data_sntv[num_seats].append((fpv_b_share, sntv_b_seat_share))

    block_plurality_election = Borda(profile, m = num_seats, tiebreak="first_place", score_vector=[1]*num_seats + [0]*(len(cand_list)-num_seats))
    block_plurality_winners = [c for s in block_plurality_election.get_elected() for c in s]
    block_plurality_num_B_winners = len([c for c in block_plurality_winners if cand_to_bloc[c] == "B"])
    block_plurality_b_seat_share = block_plurality_num_B_winners/num_seats
    block_plurality_data[num_seats].append((fpv_b_share, block_plurality_b_seat_share))


In [ ]:
from scipy.stats import gaussian_kde

rules = ["STV", "Borda", "Ranked Pairs", "SNTV", "Block Plurality"]
rule_data = {"STV": data_stv, "Borda": data_borda, "Ranked Pairs": data_ranked_pairs, "SNTV": data_sntv, "Block Plurality": block_plurality_data}


# Define colors for each rule
colors = {
    "STV": stv_color,
    "Borda": borda_color,
    "Ranked Pairs": "#7CB342",  # green
    "SNTV": "#FFA726",  # orange
    "Block Plurality": "#AB47BC"  # purple
}

plt.subplots(figsize=(10,3))

# Create KDE for each rule and plot
x_range = np.linspace(-1, 1, 500)
for rule in rules:
    data = rule_data[rule]
    values = [y-x for seats in data for (x,y) in data[seats]]
    
    if len(values) > 1:  # Need at least 2 points for KDE
        kde = gaussian_kde(values)
        density = kde(x_range)
        plt.plot(x_range, density, label=rule, color=colors[rule], alpha=0.7, linewidth=2)

plt.xlabel("S - C")
plt.legend()
plt.xlim(-1, 1)
plt.show()

In [ ]:
for rule in rules:
    print(rule, '{:.3f}'.format(np.mean([np.abs(y-x) for seats in rule_data[rule] for (x,y) in rule_data[rule][seats]])))